In [33]:
import os
import importlib
# os.environ["CUDA_VISIBLE_DEVICES"]="2,3"
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, AutoModel
from datasets import load_dataset
import torch
#from sentence_transformers import SentenceTransformer, InputExample, losses
#from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator, SimilarityFunction
from torch.utils.data import DataLoader
from datasets import Dataset
import pandas as pd
from collections import defaultdict
import re
import numpy as np
from tqdm import tqdm

data_dir = '/raid/deallab/SF_RAG_Data/ASQA'
data_dir = '../data'

device1 = 'cuda:0'
device2 = 'cuda:1'

from evaluation import evaluate
import prompts
importlib.reload(prompts)

<module 'prompts' from '/home/explorer/CCE/djk/sf_rag/sf_rag/model/prompts.py'>

In [3]:
#load embeddings
embedd_test_path = f'{data_dir}/test/embedd_test.npy'
evidence_embeddings = np.load(embedd_test_path)
print(evidence_embeddings.shape)
evidence_embeddings = torch.from_numpy(evidence_embeddings).to(device1)

#load evidence
evidence_test_path = f'{data_dir}/test/evidence_test.csv'
evidence_df = pd.read_csv(evidence_test_path)

#load qa data
qa_df=pd.read_csv(f'{data_dir}/test/qa_test.csv') #data=df[['question','long_answers']] # questions=data['question'] #references = [row.to_dict() for i, row in df.iterrows() if i < len(questions)]
qa_df.head()

(11645, 4096)


,id,sample_id,question,follow_up_questions,long_answers,short_answers
0,b11f0628-a295-410b-b63e-9a6aae0fc415,-7013890438520559398,Who has the highest goals in world football?,"[""Who has the highest goals in men's world int...","[""Ali Daei has the highest goals in men's worl...","[['Daei', 'Ali Daei'], ['Bican', 'Josef Bican'..."
1,0209e925-32e1-4607-8b4f-8c0795b93383,7089015503030534342,Who is the original artist of sound of silence?,['Who is the original artist of sound of silen...,[' The original artist of the song sound of si...,"[['Simon & Garfunkel', 'Paul Simon and Art Gar..."
2,a2d355c4-9e28-4325-83d5-45013a6d415d,8793099883447006698,When was the first apple i phone made?,"['When was the first apple i phone released?',...",['The iPhone beta was created in 2004 to test ...,"[['June 29, 2007'], ['2004'], ['June 29, 2007...."
3,3343fc27-96f0-41bf-953a-4b42bfb07bb1,-881464876144297194,Who played the weasley brothers in harry potter?,['Who played Bill weasley in Harry Potter and...,['Rupert Grint played Ron Weasley in all the H...,"[['Richard Fish'], ['Chris Rankin'], ['James P..."
4,7d7c9b56-f103-4f07-9288-aeb9f3e8560c,1650309494326541834,How many state parks are there in virginia?,['How many state parks are there in virginia i...,['When the Virginia state park system was form...,"[['six'], ['38'], ['6'], ['38']]"


In [4]:
#load quantized model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_storage=torch.bfloat16,
)

# load model with tokenizer
model = AutoModel.from_pretrained(
    'nvidia/NV-Embed-v2', 
    trust_remote_code=True,
    quantization_config = bnb_config,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage =True,
)
model.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:08<00:00,  2.22s/it]


NVEmbedModel(
  (latent_attention_model): LatentAttentionModel(
    (cross_attend_blocks): ModuleList(
      (0): PreNorm(
        (fn): Attention(
          (to_q): Linear4bit(in_features=4096, out_features=32768, bias=False)
          (to_kv): Linear4bit(in_features=4096, out_features=65536, bias=False)
          (to_out): Linear4bit(in_features=32768, out_features=4096, bias=False)
        )
        (norm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
        (norm_context): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
      )
      (1): PreNorm(
        (fn): FeedForward(
          (net): Sequential(
            (0): Linear4bit(in_features=4096, out_features=32768, bias=True)
            (1): GEGLU()
            (2): Linear4bit(in_features=16384, out_features=4096, bias=True)
          )
        )
        (norm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
      )
    )
  )
  (embedding_model): BidirectionalMistralModel(
    (embed_tokens): Embedding(

In [5]:
#load tokenizer
tokenizer_gen = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")
tokenizer_gen.pad_token = tokenizer_gen.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    # bnb_4bit_quant_type="nf4",
    # bnb_4bit_compute_dtype=torch.bfloat16,
    # bnb_4bit_use_double_quant=True,
    # bnb_4bit_quant_storage=torch.bfloat16,
)

model_gen = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map= 'auto'
)
model_gen.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.03it/s]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps

In [6]:
# retrive docs from the document embeddings
def retrieve_documents(query):
    max_length = 1024
    
    #query prefix
    task_name_to_instruct = {"example": "Given a question, retrieve passages that answer the question",}
    query_prefix = "Instruct: "+task_name_to_instruct["example"]+"\nQuery: "
    
    query_embedding = model.encode([query],instruction=query_prefix, max_length=max_length).to(device1)

    # query_embedding = query_embedding.unsqueeze(0)
    #print(query_embedding)
    similarities = torch.nn.functional.cosine_similarity(query_embedding, evidence_embeddings)

    top_results = similarities.argsort(descending=True)[:10].cpu().detach().numpy()
    #print(top_results)
    res=[evidence_df.loc[idx, 'text'] for idx in top_results if idx < len(evidence_df)]
        
    return top_results, res

In [8]:
def evaluate_docs(query, docs):
    # print(f"Query : {query}")
    # print("-"*100)
    outs = []
    for idx, doc in enumerate(docs):
        #print(f"Rank {idx} : {doc}")
        
        input= f'''
        Query: {query}
        Doc:  {doc}
        '''

        messages = [
            {"role":"user", 'content':prompts.PROMPT['eval_doc_instr']},
            {"role":"assistant", 'content':prompts.PROMPT['eval_doc_answ1']},
            {"role":"user", 'content':prompts.PROMPT['eval_doc_ex2']},
            {"role":"assistant", 'content':prompts.PROMPT['eval_doc_answ2']},
            {"role":"user", 'content':prompts.PROMPT['eval_doc_ex3']},
            {"role":"assistant", 'content':prompts.PROMPT['eval_doc_answ3']},
            {"role":"user", 'content':input}, 
        ]
        #apply tokenizter + generate eval
        inputs = tokenizer_gen.apply_chat_template(messages, return_tensors="pt", truncation=True).to(device1)
        
        attention_mask = (inputs != tokenizer_gen.pad_token_id).long().to(device1)
        
        outputs = model_gen.generate(inputs, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens=128)
        generated_text = tokenizer_gen.decode(outputs[0]).split('<|end_header_id|>')[-1].replace('<|eot_id|>', '').strip('\n')
        
        if '#relevant' in generated_text:
            outs.append(doc)
    
    return outs

In [15]:
def make_new_query(query,context):
    
    input= f'''
    Original Query: {query}
    Context information: {context}
    '''
    
    messages = [
        {"role":"user", 'content':prompts.PROMPT['refine_query_instr']},
        {"role":"assistant", 'content':prompts.PROMPT['refine_query_answ1']},
        {"role":"user", 'content':prompts.PROMPT['refine_query_ex2']},
        {"role":"assistant", 'content':prompts.PROMPT['refine_query_answ2']},
        {"role":"user", 'content':input},
    ]
    inputs = tokenizer_gen.apply_chat_template(messages, return_tensors="pt", truncation=True).to(device1)
    
    attention_mask = (inputs != tokenizer_gen.pad_token_id).long().to(device1)
    
    outputs = model_gen.generate(inputs, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens=512)
    generated_text = tokenizer_gen.decode(outputs[0]).split('<|end_header_id|>')[-1].replace('<|eot_id|>', '').strip('\n')
    generated_text  = generated_text.strip('[]').split(',\n')
    
    print(generated_text)
    return generated_text

In [25]:
# def preprocessing(new_questions):
#     return list(((new_questions.split("['")[1]).split("']")[0]).split("',\n    '"))

In [9]:
# preprocessing(make_new_query(query, rel_docs))

In [34]:
def make_new_answer(query,context):
    
    input= f'''
    Original Query: {query}
    Context information: {context}
    '''
    
    messages = [
        {"role":"user", 'content':prompts.PROMPT['new_answer_instr']},
        {"role":"assistant", 'content':prompts.PROMPT['new_answer_answ1']},
        {"role":"user", 'content':prompts.PROMPT['new_answer_ex2']},
        {"role":"assistant", 'content':prompts.PROMPT['new_answer_answ2']},
        {"role":"user", 'content':input},
    ]
    inputs = tokenizer_gen.apply_chat_template(messages, return_tensors="pt", truncation=True).to(device1)
    
    attention_mask = (inputs != tokenizer_gen.pad_token_id).long().to(device1)
    
    outputs = model_gen.generate(inputs, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens=256)
    generated_text = tokenizer_gen.decode(outputs[0]).split('<|end_header_id|>')[-1].replace('<|eot_id|>', '').strip('\n')
    
    # print(f'New Answer: {generated_text}')
    return generated_text

In [11]:
# import re

# def summarize_answers(question, answers):
#     input= f'''
#     Query: {query}
#     Context information: {answers}
#     '''
    
#     messages = [
#         {"role":"user", 'content':prompts.PROMPT['final_answer_instr']},
#         {"role":"assistant", 'content':prompts.PROMPT['final_answer_answ1']},
#         {"role":"user", 'content':{input}},
#     ]

#     #tokenizer prompt
#     input_ids = tokenizer_gen.apply_chat_template(messages, return_tensors="pt", truncation=True).to(device)

#     attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device)

#     out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
#     res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
#     candidate = [re.sub('\n|<\|eot_id\|>', '', res)]
#     return candidate

In [12]:
# data=qa_df[['question','long_answers']]
# questions=data['question']

In [13]:
# references = [row.to_dict() for i, row in qa_df.iterrows() if i < len(questions)]

In [14]:
# references[0]

In [35]:
def final_ans(query,answer, qa_pairs):
    # prompt = f"""
    # Context information is below.
    # ---------------------
    # {answers}
    # ---------------------
    # Given the context information and not prior knowledge, 
    # Answer questions that have multiple correct answers based on multiple interpretations, including multiple answers.
    # Query: {query}
    # Answer:
    # """
    
    # input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device1)
    qa_sample = '''Follow-up Query{i}: {q}
    Context: {context}
    '''
    qa_string = '\n'.join([qa_sample.format(i=i, q=q, context=a) for i, (q, a) in enumerate(qa_pairs)])
    
    input= f'''
    Initial Query: {query}
    Context: {answer}
    {qa_string}
    '''
    print(f'Final Answ Input:{input}')
    messages = [
        {"role":"user", 'content':prompts.PROMPT['final_answer_instr']},
        {"role":"assistant", 'content':prompts.PROMPT['final_answer_answ1']},
        {"role":"user", 'content':input},
    ]

    #tokenizer prompt
    input_ids = tokenizer_gen.apply_chat_template(messages, return_tensors="pt", truncation=True).to(device2)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device2)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    candidate = [re.sub('\n|<\|eot_id\|>', '', res)]
    #print(candidate)
    return candidate

In [36]:
from evaluation import evaluate
from collections import defaultdict

# sf_rag=dict()
# perplexity_df=pd.DataFrame()
scores_list=[]
stop_iteration = 90
new_answers_dic=defaultdict(list)

for idx, row in tqdm(qa_df.iterrows(), total=min(len(qa_df), stop_iteration)):
    if idx == stop_iteration: break
    query = row['question']
    
    #retrieve relevant docs
    ids, docs = retrieve_documents(query)
    rel_docs = evaluate_docs(query, docs)
    answer = make_new_answer(query, rel_docs)
    print(f'Initial Answer: {answer}')
    
    # generate new queries
    new_queries=make_new_query(query, rel_docs)
    
    #iterate over new docs
    qa_pairs = []
    for i, new_query in tqdm(enumerate(new_queries)):
        if i == 5: break #brak after x follow-up question 
        
        # retrieve relevant docs
        ids, new_docs=retrieve_documents(new_query)
        new_rel_docs=evaluate_docs(new_query, new_docs)
        new_answer = make_new_answer(new_query, new_rel_docs)
        qa_pairs.append((new_query, new_answer))

    # generate final answer
    candidate=final_ans(query, answer, qa_pairs)
    print(f'candidate: {candidate}')
    # print(references[i])
    scores=evaluate(candidate,[row.to_dict()])
    print(scores)
    scores_list.append(scores)
    scores_df=pd.DataFrame(scores_list)
    scores_df.to_csv('./results/self-refine_results.csv', index=False)
    
scores_df=pd.DataFrame(scores_list)
print(scores_df.mean())
scores_df.to_csv('./results/self-refine_results.csv', index=False)

  0%|          | 0/90 [00:00<?, ?it/s]

Initial Answer: The highest goalscorer in world football is Cristiano Ronaldo with 133 international goals.
["'### Who holds the record for the most goals scored in a single game of the ncaa football national championship?'", "'### Which team has the most appearances in the ncaa football national championship?'", "'### Who is the only player to have won the ncaa football national championship in multiple decades?'", "'### What is the highest score ever recorded in the ncaa football national championship?'", "'### Who are the top scorers in the ncaa football national championship?'", "'### What is the average number of goals scored per game in the ncaa football national championship?'", "'### How many players have scored 10 or more goals in a single season in the ncaa football national championship?'", "'### What is the record for most goals scored by a single player in a single season in the ncaa football national championship?'", "'### Who is the only player to have scored a goal in e

5it [00:43,  8.64s/it]


Final Answ Input:
    Initial Query: Who has the highest goals in world football?
    Context: The highest goalscorer in world football is Cristiano Ronaldo with 133 international goals.
    Follow-up Query0: '### Who holds the record for the most goals scored in a single game of the ncaa football national championship?'
    Context: There is no information provided about the NCAA football national championship. The context information is empty and does not provide any information about goals scored in a single game of the NCAA football national championship.
    
Follow-up Query1: '### Which team has the most appearances in the ncaa football national championship?'
    Context: The University of Alabama has the most appearances in the NCAA football national championship.
    
Follow-up Query2: '### Who is the only player to have won the ncaa football national championship in multiple decades?'
    Context: The only player to have won the NCAA football national championship in multiple

  1%|          | 1/90 [01:21<2:00:52, 81.48s/it]

...

candidate: ['Jeanne Calment of France lived to be the oldest person in the world, with a verified age of 122 years and 164 days.']
{'rougeLsum': 21.84873949579832, 'length': 23.0, 'str_em': 40.0, 'ovscore': 29.562638242077327}
Initial Answer: The 12th day of Christmas begins on January 5th, and the entire period of the Twelve Days of Christmas is from December 25th to January 5th, counting the first and last day.
["'### What is the beginning date of the 12th day of Christmas in the Gregorian calendar?'", "'### What is the end date of the 12th day of Christmas in the Julian calendar?'", "'### When does the 12th day of Christmas begin in the Armenian Apostolic Church and Armenian Catholic Church?'", "'### Which Christian denominations celebrate the 12th day of Christmas differently?'", "'### What is the significance of the 12th day of Christmas in the context of the Epiphany feast?'", "'### How does the 12th day of Christmas relate to the traditional liturgical seasons of Advent and Chri

5it [02:27, 29.41s/it]


Final Answ Input:
    Initial Query: When does the 12th day of christmas begin?
    Context: The 12th day of Christmas begins on January 5th, and the entire period of the Twelve Days of Christmas is from December 25th to January 5th, counting the first and last day.
    Follow-up Query0: '### What is the beginning date of the 12th day of Christmas in the Gregorian calendar?'
    Context: The beginning date of the 12th day of Christmas in the Gregorian calendar is January 5th, but considering the context, the 12th day is the day before, which is January 4th is not considered, but rather January 5th is the end of the 12th day, the actual beginning date is January 5th - 1 day, which is January 4th is not considered, but rather the 12 days start on December 25th, and the 12th day is January 5th, so the 12 days are from December 25th to January 5th.
    
Follow-up Query1: '### What is the end date of the 12th day of Christmas in the Julian calendar?'
    Context: The 12th day of Christmas i

 48%|████▊     | 43/90 [1:54:12<2:03:34, 157.75s/it]

candidate: ['The 12th day of Christmas begins on December 25th and ends on January 5th, with the 12th day being January 5th.']
{'rougeLsum': 24.56140350877193, 'length': 21.0, 'str_em': 40.0, 'ovscore': 31.34415639877515}
Initial Answer: There is no information provided regarding Alabama becoming a right-to-work state.
["'### When did Alabama become a state?'", "'### What is the current labor law status in Alabama?'", "'### What is the history of labor laws in Alabama?'", "'### Did Alabama ever have a law that prohibited right to work laws?'", "'### What is the current status of labor unions in Alabama?'", "'### When did the Alabama state government last pass a law that affected labor rights?'", "'### What are the current labor laws in Alabama regarding unionization?'", "'### Did Alabama ever have a law that prohibited at-will employment?'", "'### What is the current status of right to work laws in the United States?'", "'### How does the right to work law in Alabama affect labor union

In [ ]:
#  20%|██        | 1/5 [01:19<05:16, 79.10s/it]
# ['Based on the provided context information, there are multiple answers to the query "Who has the highest goals in world football?" depending on the interpretation.1. **Cristiano Ronaldo**: With 133 international goals, Cristiano Ronaldo holds the record for the highest number of goals scored in international football.2. **Cristiano Ronaldo (in European football)**: Ronaldo also holds the record for the highest number of goals scored in European football, with 85 international goals.3. **Cristiano Ronaldo (in European Championship)**: He is the first player to score 14 goals at the European Championships.4. **Cristiano Ronaldo (in UEFA Nations League)**: Ronaldo is the top scorer in the inaugural UEFA Nations League, with 5 goals.5. **Pelé**: He was the first player from South America to score at least 50 international goals and went on to score 77 international goals in 92 matches.6. **Mokhtar Dahari**: He broke the record for the highest international goalscorer, scoring 89 goals for Malaysia in 142 international appearances.7. **Imre Schlosser**: He was the first player to score 50 international goals and held the record for 26 years until Ferenc Puskás broke it.8. **Ferenc Puskás**: He broke the record for the highest international goalscorer, scoring 84 goals in his international career.9. **Vivian Woodward**: He was the fastest to achieve the feat of 50 international goals, scoring his 50th goal in his 32nd official international match.10. **Lionel Messi**: He became the third player to reach and pass the milestone of 100 international goals, as well as the first South American to achieve the feat.These are just a few examples of players who have achieved significant milestones in international football.']
# {'rougeLsum': 27.368421052631582, 'length': 264.0, 'str_em': 0.0, 'ovscore': 0.0}
#  40%|████      | 2/5 [02:18<03:22, 67.38s/it]
# ['The original artist of "The Sound of Silence" is Simon & Garfunkel, specifically Paul Simon, who wrote the song, and Art Garfunkel, who sang the melody.']
# {'rougeLsum': 41.463414634146346, 'length': 26.0, 'str_em': 66.66666666666666, 'ovscore': 52.57592264788534}
#  60%|██████    | 3/5 [03:09<02:00, 60.04s/it]
# ['The development of the first Apple iPhone began in 2005, and it was officially announced on January 9, 2007.']
# {'rougeLsum': 28.915662650602407, 'length': 19.0, 'str_em': 0.0, 'ovscore': 0.0}
#  80%|████████  | 4/5 [04:01<00:56, 56.98s/it]
# ['The Weasley brothers were portrayed by James and Oliver Phelps, who played Fred and George Weasley respectively.']
# {'rougeLsum': 19.607843137254903, 'length': 17.0, 'str_em': 16.666666666666664, 'ovscore': 18.07753815155468}
#  80%|████████  | 4/5 [04:12<01:03, 63.23s/it]

In [22]:
scores_df=pd.DataFrame(scores_list)

In [23]:
scores_df.mean()

# rougeLsum    29.178478
# length       93.666667
# str_em       30.555556
# ovscore      18.200875
# dtype: float64


rougeLsum    29.338835
length       81.500000
str_em       20.833333
ovscore      17.663365
dtype: float64